In [1]:
import time
import numpy as np
from tscglue import data_loader
from tscglue.interval_models import RSTSFRandomTransformer

X_train, y_train, X_test, y_test = data_loader.load_fold("ElectricDevices", fold=0)
print(f"train={X_train.shape}  test={X_test.shape}")

train=(8926, 1, 96)  test=(7711, 1, 96)


In [ ]:
results = {}

for mode in ["fast", "default"]:
    t = RSTSFRandomTransformer(n_intervals=600, mode=mode, random_state=42, n_jobs=-1, verbose=True)

    t0 = time.perf_counter()
    Xt_train = t.fit_transform(X_train)
    fit_time = time.perf_counter() - t0

    t0 = time.perf_counter()
    Xt_test = t.transform(X_test)
    transform_time = time.perf_counter() - t0

    results[mode] = dict(fit=fit_time, transform=transform_time, shape=Xt_train.shape, mb=Xt_train.nbytes/1024**2)
    print(f"\n[mode={mode}] fit_transform : {fit_time:.2f}s  -> {Xt_train.shape}  ({Xt_train.nbytes/1024**2:.1f} MB)")
    print(f"[mode={mode}] transform     : {transform_time:.2f}s  -> {Xt_test.shape}")

In [ ]:
print("=== Summary ===")
print(f"{'mode':<10} {'fit (s)':>10} {'transform (s)':>15} {'speedup fit':>12} {'speedup transform':>18}")
base_fit = results["default"]["fit"]
base_tr  = results["default"]["transform"]
for mode, r in results.items():
    sf = base_fit / r["fit"]
    st = base_tr  / r["transform"]
    print(f"{mode:<10} {r['fit']:>10.2f} {r['transform']:>15.2f} {sf:>12.2f}x {st:>18.2f}x")